# Solución P03 · Calidad, joins y consultas visibles

[Enunciado](../talleres/P03.md). Ejecuta primero [P02](P02_solucion.ipynb); este notebook lee su producto, no lo genera silenciosamente. Preparar también DIVIPOLA con `python 00_datos.py --descargar divipola` desde `kit/`. Si falta una entrada, detenerse y prepararla.

Seleccionar el kernel del curso. Las reglas se escriben con pandas, las uniones con `merge` y las consultas posteriores con las API de Polars y DuckDB. No se llama a funciones del kit. Ejecutar una celda a la vez y explicar la evidencia antes de decidir qué publicar.

In [1]:
import pandas as pd
import os
import json

repositorio = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
archivo_parquet = repositorio + "/kit/salidas/pandas/P02/pqrs.parquet"
archivo_divipola = repositorio + "/kit/data/raw/divipola.csv"
salida = repositorio + "/kit/salidas/pandas/P03"
print("Entrada:", archivo_parquet)
print("Catálogo territorial:", archivo_divipola)
assert os.path.isfile(archivo_parquet), "Ejecutar P02 antes de P03"
assert os.path.isfile(archivo_divipola), "Preparar DIVIPOLA según el enunciado"
df_pqrs = pd.read_parquet(archivo_parquet)
display(df_pqrs.head())
df_pqrs.info()
assert df_pqrs.shape == (3000, 16)

Entrada: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/pandas/P02/pqrs.parquet
Catálogo territorial: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/raw/divipola.csv


,corte_archivo,fila_fuente,OBJECTID,periodo,mes,afec_cod_depto,afec_cod_mpio,pet_cod_depto,pet_cod_mpio,ent_nombre,afec_edad,afec_genero,anio,mes_num,edad_num,clave_candidata
0,2023_II,2,1,2023,7,11,11001,11,11001,NUEVA EPS,69,HOMBRE,2023,7,69.0,2023_II|1
1,2023_II,3,2,2023,7,11,11001,11,11001,SANITAS,35,MUJER,2023,7,35.0,2023_II|2
2,2023_II,4,3,2023,7,70,70702,70,70702,MAGISTERIO,77,MUJER,2023,7,77.0,2023_II|3
3,2023_II,5,4,2023,7,76,76892,76,76892,ASMET SALUD EPS,71,HOMBRE,2023,7,71.0,2023_II|4
4,2023_II,6,5,2023,7,08,08001,08,08001,CAJACOPI EPS,48,MUJER,2023,7,48.0,2023_II|5


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   corte_archivo    3000 non-null   object 
 1   fila_fuente      3000 non-null   int64  
 2   OBJECTID         3000 non-null   object 
 3   periodo          3000 non-null   object 
 4   mes              3000 non-null   object 
 5   afec_cod_depto   3000 non-null   object 
 6   afec_cod_mpio    3000 non-null   object 
 7   pet_cod_depto    3000 non-null   object 
 8   pet_cod_mpio     3000 non-null   object 
 9   ent_nombre       3000 non-null   object 
 10  afec_edad        3000 non-null   object 
 11  afec_genero      3000 non-null   object 
 12  anio             3000 non-null   Int64  
 13  mes_num          3000 non-null   Int64  
 14  edad_num         3000 non-null   Float64
 15  clave_candidata  3000 non-null   object 
dtypes: Float64(1), Int64(2), int64(1), object(12)
memory usage: 

## 1. Escribir reglas e inspeccionar las filas
Separamos identificadores, período, edad y ubicación. Las reglas de revisión no eliminan registros. Usamos `fillna` en las máscaras para que un valor ausente no oculte una incidencia.

In [2]:
id_vacio = df_pqrs["OBJECTID"].fillna("").str.strip().eq("")
periodo_revision = df_pqrs["anio"].isna() | df_pqrs["mes_num"].isna() | ~df_pqrs["mes_num"].between(1, 12).fillna(False)
edad_revision = df_pqrs["edad_num"].isna() | df_pqrs["edad_num"].lt(0).fillna(False)
mayores_90 = df_pqrs["edad_num"].gt(90).fillna(False)
peticionario_vacio = (df_pqrs["pet_cod_depto"].fillna("").str.strip().eq("")
                     | df_pqrs["pet_cod_mpio"].fillna("").str.strip().eq(""))
clave_repetida = df_pqrs.duplicated(["corte_archivo", "OBJECTID"], keep=False)
control = pd.Series({"filas": len(df_pqrs), "claves_candidatas": df_pqrs["clave_candidata"].nunique(),
                     "id_vacio": int(id_vacio.sum()), "periodo_en_revision": int(periodo_revision.sum()),
                     "edad_en_revision": int(edad_revision.sum()), "mayores_90": int(mayores_90.sum()),
                     "ubicacion_peticionario_vacia": int(peticionario_vacio.sum()),
                     "filas_clave_repetida": int(clave_repetida.sum()), "exclusiones_automaticas": 0})
display(control)
display(df_pqrs.loc[mayores_90, ["corte_archivo", "OBJECTID", "afec_edad", "edad_num"]].head(10))
display(df_pqrs.loc[peticionario_vacio, ["OBJECTID", "pet_cod_mpio", "afec_cod_mpio"]])
assert control["mayores_90"] == 41 and control["ubicacion_peticionario_vacia"] == 3
assert control["claves_candidatas"] == 3000

filas                           3000
claves_candidatas               3000
id_vacio                           0
periodo_en_revision                0
edad_en_revision                   0
mayores_90                        41
ubicacion_peticionario_vacia       3
filas_clave_repetida               0
exclusiones_automaticas            0
dtype: int64

,corte_archivo,OBJECTID,afec_edad,edad_num
252,2023_II,253,98,98.0
288,2023_II,289,92,92.0
380,2023_II,381,91,91.0
419,2023_II,420,91,91.0
460,2023_II,461,92,92.0
545,2023_II,546,99,99.0
547,2023_II,548,106,106.0
606,2023_II,607,92,92.0
623,2023_II,624,92,92.0
713,2023_II,714,98,98.0


,OBJECTID,pet_cod_mpio,afec_cod_mpio
717,718,,20228
953,954,,44078
1239,240,,5001


Edad mayor de 90 no prueba error; un campo territorial vacío no prueba anonimato. Una proyección de 16 campos tampoco permite detectar duplicados exactos de los 38 originales. ¿Qué fuente adicional necesitarías antes de eliminar una fila?

## 2. Leer y preparar la dimensión territorial
Los códigos son texto. Revisamos su formato **antes** de completar ceros. `validate="many_to_one"` exige que cada código de la dimensión aparezca una sola vez: así un catálogo duplicado no multiplica reportes.

In [3]:
df_divipola = pd.read_csv(archivo_divipola, sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
display(df_divipola.head())
dimension = df_divipola[["Código Municipio", "Nombre Departamento", "Nombre Municipio"]].copy()
dimension = dimension.rename(columns={"Código Municipio": "codigo_municipio",
    "Nombre Departamento": "departamento_catalogo", "Nombre Municipio": "municipio_catalogo"})
codigo = dimension["codigo_municipio"].str.strip()
mal_formado = ~codigo.str.fullmatch(r"[0-9]{1,5}", na=False)
display(dimension.loc[mal_formado])
assert not mal_formado.any(), "Revisar códigos del catálogo"
dimension["codigo_municipio"] = codigo.str.zfill(5)
display(dimension.loc[dimension.duplicated("codigo_municipio", keep=False)])
assert dimension["codigo_municipio"].is_unique

,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud
0,05,ANTIOQUIA,05001,MEDELLÍN,Municipio,"-75,581775","6,246631"
1,05,ANTIOQUIA,05002,ABEJORRAL,Municipio,"-75,428739","5,789315"
2,05,ANTIOQUIA,05004,ABRIAQUÍ,Municipio,"-76,064304","6,632282"
3,05,ANTIOQUIA,05021,ALEJANDRÍA,Municipio,"-75,141346","6,376061"
4,05,ANTIOQUIA,05030,AMAGÁ,Municipio,"-75,702188","6,038708"


,codigo_municipio,departamento_catalogo,municipio_catalogo


,codigo_municipio,departamento_catalogo,municipio_catalogo


## 3. Left join del afectado y anti-join
Conservamos el código original y creamos una llave separada. Los códigos no válidos quedan pendientes; no inventamos correspondencias. `indicator=True` muestra qué filas encontraron coincidencia. El anti-join se obtiene filtrando `left_only`.

In [4]:
df_trabajo = df_pqrs.copy()
codigo_afectado = df_trabajo["afec_cod_mpio"].astype("string").str.strip()
formato_afectado = codigo_afectado.str.fullmatch(r"[0-9]{1,5}", na=False)
df_trabajo["llave_afectado"] = codigo_afectado.where(formato_afectado).str.zfill(5)
df_afectado = df_trabajo.merge(dimension, how="left", left_on="llave_afectado",
                             right_on="codigo_municipio", validate="many_to_one", indicator=True)
assert len(df_afectado) == len(df_pqrs)
sin_afectado = df_afectado.loc[df_afectado["_merge"].eq("left_only")].copy()
display(df_afectado["_merge"].value_counts())
display(sin_afectado[["corte_archivo", "OBJECTID", "afec_cod_mpio", "llave_afectado"]].head(20))

_merge
both          3000
left_only        0
right_only       0
Name: count, dtype: int64

,corte_archivo,OBJECTID,afec_cod_mpio,llave_afectado


## 4. Repetir el join para el peticionario
No sustituimos al peticionario por el afectado: son roles distintos. Modifica una copia de un código y observa cómo aparece en el anti-join, sin cambiar los originales.

In [5]:
codigo_peticionario = df_trabajo["pet_cod_mpio"].astype("string").str.strip()
formato_peticionario = codigo_peticionario.str.fullmatch(r"[0-9]{1,5}", na=False)
df_trabajo["llave_peticionario"] = codigo_peticionario.where(formato_peticionario).str.zfill(5)
df_peticionario = df_trabajo.merge(dimension, how="left", left_on="llave_peticionario",
                                 right_on="codigo_municipio", validate="many_to_one", indicator=True)
assert len(df_peticionario) == 3000
sin_peticionario = df_peticionario.loc[df_peticionario["_merge"].eq("left_only")].copy()
display(df_peticionario["_merge"].value_counts())
display(sin_peticionario[["corte_archivo", "OBJECTID", "pet_cod_mpio", "llave_peticionario"]].head(20))

_merge
both          2997
left_only        3
right_only       0
Name: count, dtype: int64

,corte_archivo,OBJECTID,pet_cod_mpio,llave_peticionario
717,2023_II,718,,<NA>
953,2023_II,954,,<NA>
1239,2024_I,240,,<NA>


**Interpretación territorial:** un código sin coincidencia requiere investigar formato, vigencia o cobertura del catálogo. Conservar filas sin correspondencia. No tratar un join exitoso como prueba de equivalencia histórica de límites.

## 5. Provocar una multiplicación y detenerla
Añadimos una fila duplicada únicamente a una copia de la dimensión. Observa el error que impide publicar un join que podría multiplicar filas.

In [6]:
dimension_defectuosa = pd.concat([dimension, dimension.iloc[[0]]], ignore_index=True)
try:
    df_trabajo.merge(dimension_defectuosa, how="left", left_on="llave_afectado",
                    right_on="codigo_municipio", validate="many_to_one")
except pd.errors.MergeError as error:
    print("Join detenido:", error)
else:
    raise AssertionError("Se esperaba un error de cardinalidad")

Join detenido: Merge keys are not unique in right dataset; not a many-to-one merge


## 6. Filtrar y agrupar con pandas
La consulta limita el universo a 2024. Agrupamos **reportes** por corte y mes; `.size()` cuenta filas, mientras `.count()` omite nulos por columna. El filtro no elimina filas del dataset original.

In [7]:
df_2024 = df_pqrs.loc[df_pqrs["anio"].eq(2024)].copy()
resumen_pandas = (df_2024.groupby(["corte_archivo", "mes_num"], dropna=False)
                  .size().rename("reportes").reset_index()
                  .sort_values(["corte_archivo", "mes_num"]).reset_index(drop=True))
display(resumen_pandas)
assert resumen_pandas["reportes"].sum() == len(df_2024) == 2000
assert len(df_pqrs) == 3000

,corte_archivo,mes_num,reportes
0,2024_I,1,11
1,2024_I,2,9
2,2024_I,3,17
3,2024_I,4,47
4,2024_I,5,188
5,2024_I,6,728
6,2024_II,7,1000


## 7. La misma consulta con la API diferida de Polars
`scan_parquet` construye un plan, `explain` lo muestra y `collect` ejecuta. Busca el filtro y la selección de columnas en el plan. El modo streaming no garantiza que cualquier operación quepa fuera de memoria.

In [8]:
import polars as pl
consulta = (pl.scan_parquet(archivo_parquet)
    .filter(pl.col("anio") == 2024)
    .group_by(["corte_archivo", "mes_num"])
    .agg(pl.len().alias("reportes")))
print(consulta.explain())
resumen_polars = (consulta.collect(engine="streaming").to_pandas()
    .sort_values(["corte_archivo", "mes_num"]).reset_index(drop=True))
display(resumen_polars)
pd.testing.assert_frame_equal(resumen_pandas, resumen_polars, check_dtype=False)

AGGREGATE[maintain_order: false]
  [len().alias("reportes")] BY [col("corte_archivo"), col("mes_num")]
  FROM
  Parquet SCAN [/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/pandas/P02/pqrs.parquet]
  PROJECT 3/16 COLUMNS
  SELECTION: [(col("anio")) == (2024)]
  ESTIMATED ROWS: 3000


,corte_archivo,mes_num,reportes
0,2024_I,1,11
1,2024_I,2,9
2,2024_I,3,17
3,2024_I,4,47
4,2024_I,5,188
5,2024_I,6,728
6,2024_II,7,1000


## 8. SQL sobre el mismo archivo
DuckDB lee Parquet directamente. Parametrizamos la ruta y cerramos la conexión. Comparar todos los grupos detecta diferencias que el total por sí solo puede ocultar.

In [9]:
import duckdb
with duckdb.connect() as conexion:
    resumen_sql = conexion.execute("""
        SELECT corte_archivo, mes_num, count(*) AS reportes
        FROM read_parquet(?)
        WHERE anio = 2024
        GROUP BY corte_archivo, mes_num
        ORDER BY corte_archivo, mes_num
    """, [archivo_parquet]).df()
display(resumen_sql)
pd.testing.assert_frame_equal(resumen_pandas, resumen_sql, check_dtype=False)
assert resumen_sql["reportes"].sum() == 2000

,corte_archivo,mes_num,reportes
0,2024_I,1,11
1,2024_I,2,9
2,2024_I,3,17
3,2024_I,4,47
4,2024_I,5,188
5,2024_I,6,728
6,2024_II,7,1000


## 9. Escribir evidencia y verificarla al releer
Guardar controles y anti-joins, no un dataset del que se borraron incidencias. Repetir esta celda reemplaza los archivos de P03. Antes de escribir, observa la carpeta impresa.

In [10]:
print("Guardar en:", salida)
os.makedirs(salida, exist_ok=True)
control.to_csv(salida + "/control_calidad.csv", index_label="control", header=["cantidad"])
sin_afectado.to_csv(salida + "/afectados_sin_catalogo.csv", index=False)
sin_peticionario.to_csv(salida + "/peticionarios_sin_catalogo.csv", index=False)
resumen_pandas.to_csv(salida + "/reportes_2024.csv", index=False)
resumen_pandas.to_parquet(salida + "/reportes_2024.parquet", index=False)
comprobacion = pd.read_parquet(salida + "/reportes_2024.parquet")
pd.testing.assert_frame_equal(resumen_pandas, comprobacion)
display(comprobacion)

Guardar en: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/pandas/P03


,corte_archivo,mes_num,reportes
0,2024_I,1,11
1,2024_I,2,9
2,2024_I,3,17
3,2024_I,4,47
4,2024_I,5,188
5,2024_I,6,728
6,2024_II,7,1000


## Decisión y entrega

| Evidencia | Decisión de referencia |
|---|---|
| Esquema válido y filas conciliadas | Publicar con límites documentados |
| Edad alta, códigos ausentes o sin correspondencia | Revisar fuente y catálogo; conservar registros |
| Cardinalidad incorrecta o pérdida de filas | Detener, corregir y repetir los controles |

Explica el impacto de cada decisión: en esta práctica hay **cero exclusiones automáticas**. Entrega notebook ejecutado, controles y anti-joins, junto con la comparación pandas/Polars/SQL. Los reportes PQRS no miden prevalencia ni riesgo clínico. Cambia un filtro, anticipa el resultado y verifica la equivalencia en los tres motores.

## Referencia de interpretación

La solución conserva visibles las operaciones y sus controles. No debe sustituirse por una llamada a un script: modifica una copia de una condición, predice el resultado y comprueba la evidencia. Explica las limitaciones y conserva los originales.